# (1) Business Understanding

- **Tujuan** : Memberikan rekomendasi berdasarkan hasil analisis performa penjualan dan operasional bisnis E-Commerce.

- **Permasalahan** : Bisnis E-Commerce memiliki berbagai transaksi penjualan, tetapi belum mengetahui secara mendalam bagaimana tren total pembayaran, kategori produk yang paling banyak dipesan, wilayah dengan nilai transaksi terbesar, serta pola diskon, metode pembayaran, dan pembatalan pesanan.

- **Objective Task** :
  - Bagaimana tren total pembayaran E-Commerce berdasarkan periode transaksi?
  - Kategori produk apa yang memiliki jumlah pesanan terbanyak?
  - Wilayah atau provinsi mana yang menghasilkan nilai transaksi terbesar?
  - Bagaimana pola diskon, metode pembayaran, dan pembatalan pesanan?

**Catatan:** Insight dan rekomendasi bisnis disajikan pada bagian (8), bukan sebagai objective tersendiri.

# (2) Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests


sns.set()

# (3) Import Dataset

In [ ]:
# GitHub API URL
api_url = "https://api.github.com/repos/BakiTacos/Indonesia-Ecommerce-Sales-Shipping-Data-2023-2025/contents/RAW_PUBLIC"

# Mengambil daftar file
response = requests.get(api_url)
response.raise_for_status()

# Memfilter file Excel
files = [
    item for item in response.json()
    if item["name"].endswith(".xlsx")
]

print("Total Dataset:", len(files))

In [ ]:
# Membaca seluruh file Excel dari GitHub
dataframes = {}

for file in files:
    file_name = file["name"]
    file_url = file["download_url"]

    dataframes[file_name] = pd.read_excel(file_url)

print("Berhasil membaca:", len(dataframes), "file")

In [ ]:
# Menampilkan lima baris pertama dari salah satu dataset
df = list(dataframes.values())[0]

# Menampilkan lima baris pertama dataset
df.head()

## (3.1) Preview Dataset
Melakukan micro analysis pada tahap preview dataset untuk memahami struktur dan karakteristik awal dari 24 dataset yang digunakan. Pemeriksaan dilakukan dengan membandingkan jumlah baris, jumlah kolom, serta nama kolom pada setiap dataset guna memastikan konsistensi data sebelum dilakukan penggabungan dan analisis lebih lanjut.

In [ ]:
# Memeriksa struktur setiap dataset

for file_name, data in dataframes.items():

    print("\nDataset:", file_name)
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())

In [ ]:
# Membandingkan struktur kolom setiap dataset

reference_columns = list(dataframes.values())[0].columns

for file_name, data in dataframes.items():

    if list(data.columns) != list(reference_columns):

        print("\nDataset:", file_name)
        print("Jumlah Kolom:", len(data.columns))

        print("Kolom Tambahan:",
              set(data.columns) - set(reference_columns))

        print("Kolom Hilang:",
              set(reference_columns) - set(data.columns))

**Hasil Preview Dataset:**

Berdasarkan pemeriksaan terhadap 24 dataset, ditemukan tiga file dengan struktur kolom yang berbeda. Dataset December 2024 dan July 2025 memiliki perbedaan nama kolom pada informasi waktu, sedangkan dataset September 2024 memiliki 49 kolom dengan informasi tambahan terkait produk, pembeli, diskon, dan pengiriman. Perbedaan struktur tersebut perlu diperiksa lebih lanjut untuk menentukan penyesuaian data sebelum proses penggabungan dilakukan.

## (3.2) Additional Dataset Inspection

Penjelasan:

Melakukan pemeriksaan lebih lanjut terhadap dataset September 2024 yang memiliki jumlah kolom lebih banyak dibandingkan dataset lainnya. Pemeriksaan bertujuan mengidentifikasi informasi tambahan dan menentukan relevansinya terhadap kebutuhan analisis.

In [ ]:
# Menampilkan seluruh kolom dataset September 2024

df_september = dataframes['SeptemberSales2024_public.xlsx']

for i, column in enumerate(df_september.columns, 1):
    print(f"{i}. {column}")

**Hasil Pemeriksaan Dataset September 2024:**

Berdasarkan pemeriksaan lebih lanjut, dataset September 2024 memiliki 49 kolom, sedangkan mayoritas dataset lainnya memiliki 17 kolom. Kolom tambahan mencakup informasi detail produk, harga, diskon, pembayaran, dan pengiriman. Selain itu, terdapat beberapa kolom yang mengandung informasi pribadi pelanggan sehingga tidak digunakan dalam analisis publik. Untuk menjaga konsistensi analisis antarperiode, kolom yang digunakan pada analisis utama akan disesuaikan dengan variabel yang tersedia pada seluruh dataset.

In [ ]:
# Menampilkan nama kolom dataset standar

for file_name, data in dataframes.items():

    if file_name != 'SeptemberSales2024_public.xlsx':

        print("\nDataset:", file_name)
        print("Columns:", data.columns.tolist())

**Hasil Pemeriksaan Konsistensi Dataset:**

Berdasarkan pemeriksaan terhadap 24 dataset, ditemukan 3 dataset yang memiliki perbedaan struktur kolom. Dataset December 2024 dan July 2025 memiliki perbedaan pada kolom waktu, sedangkan September 2024 memiliki 49 kolom dengan 32 kolom tambahan dibandingkan struktur standar. Sementara itu, 21 dataset lainnya memiliki struktur 17 kolom yang konsisten. Oleh karena itu, diperlukan penyesuaian lebih lanjut sebelum seluruh dataset digabungkan untuk proses analisis.

# (4) Data Understanding

Pada tahap Data Understanding dilakukan pemeriksaan terhadap dataset yang telah dipilih untuk memahami struktur, tipe data, serta jumlah baris dan kolom. Tahap ini bertujuan memperoleh gambaran umum mengenai karakteristik dataset sebelum dilakukan proses Data Preprocessing.

In [ ]:
# Menggabungkan 24 dataset dengan penyesuaian kolom tanggal

standard_columns = list(dataframes['AprilSales2024_public.xlsx'].columns)

prepared_data = []

for file_name, data in dataframes.items():
    temp = data.copy()

    # Menentukan tanggal analisis dan mencatat sumber tanggalnya
    if 'Waktu Pesanan Dibuat' in temp.columns:
        temp['Tanggal Analisis'] = temp['Waktu Pesanan Dibuat']
        temp['Sumber Tanggal'] = 'Waktu Pesanan Dibuat'
    elif 'Waktu Pengiriman Diatur' in temp.columns:
        temp['Tanggal Analisis'] = temp['Waktu Pengiriman Diatur']
        temp['Sumber Tanggal'] = 'Waktu Pengiriman Diatur'
    else:
        temp['Tanggal Analisis'] = pd.NA
        temp['Sumber Tanggal'] = 'Tidak tersedia'

    temp = temp.reindex(columns=standard_columns + ['Tanggal Analisis', 'Sumber Tanggal'])
    temp['source_file'] = file_name

    prepared_data.append(temp)

df = pd.concat(prepared_data, ignore_index=True)

df.head()

In [ ]:
df.info()

In [ ]:
print('baris dataframe berjumlah : ', df.shape[0])
print('kolom dataframe berjumlah : ', df.shape[1])

Catatan :
- Dataset merupakan gabungan dari 24 file Excel transaksi E-Commerce.
- Dataset gabungan memiliki 26.258 baris sebelum proses pembersihan duplikasi.
- Kolom `source_file` digunakan untuk mengidentifikasi file asal setiap baris.
- Kolom `Tanggal Analisis` digunakan untuk analisis tren bulanan.
- Sebagian besar file menggunakan `Waktu Pesanan Dibuat`. Khusus file Desember 2024 dan Juli 2025, tanggal alternatif yang tersedia adalah `Waktu Pengiriman Diatur`.
- Perbedaan sumber tanggal tersebut dicatat pada kolom `Sumber Tanggal` agar keterbatasan analisis periode dapat diketahui.
- Pemeriksaan tipe data, nilai kosong, dan duplikasi dilakukan pada tahap Data Preprocessing.

#  (5) Data Preprocessing

Pada tahap Data Preprocessing dilakukan proses pemeriksaan dan persiapan data sebelum digunakan dalam analisis. Tahap ini bertujuan untuk memastikan data memiliki kualitas yang baik, konsisten, dan siap digunakan pada proses Exploratory Data Analysis (EDA).

In [ ]:
df.head(2)

## (5.1) Preprocessing pada kolom Kota/Kabupaten

In [ ]:
print(' Sebelum Preprocessing : \n')
pd.Series(df['Kota/Kabupaten'].unique())

In [ ]:
df['Kota/Kabupaten'] = df['Kota/Kabupaten'].apply(lambda x : x.title())

In [ ]:
print(' Sesudah Preprocessing : \n')
pd.Series(df['Kota/Kabupaten'].unique())

## (5.2) Preprocessing pada kolom Provinsi

In [ ]:
print(' Sebelum Preprocessing : \n')
pd.Series(df['Provinsi'].unique())

In [ ]:
df['Provinsi'] = df['Provinsi'].apply(lambda x : x.title())

In [ ]:
print(' Sesudah Preprocessing : \n')
pd.Series(df['Provinsi'].unique())

Catatan :
- Pada kolom Provinsi, dilakukan perubahan format penulisan menggunakan fungsi `.title()` agar setiap kata diawali huruf kapital.
- Sebelum preprocessing terdapat 34 provinsi unik dengan format huruf kapital seluruhnya.
- Setelah preprocessing, jumlah provinsi unik tetap 34 dan format penulisannya menjadi Title Case.

## (5.3) Preprocessing pada kolom Total Berat

In [ ]:
print(' Sebelum Preprocessing : \n')
pd.Series(df['Total Berat'].unique())

In [ ]:
df['Total Berat'] = df['Total Berat'].str.replace(' gr', '', regex=False).astype(float)

In [ ]:
print(' Sesudah Preprocessing : \n')
pd.Series(df['Total Berat'].unique())

## (5.4) Preprocessing pada kolom Tanggal Analisis

In [ ]:
print('Sebelum Preprocessing:')
print('Tipe Data:', df['Tanggal Analisis'].dtype)
print('Jumlah Missing Value:', df['Tanggal Analisis'].isnull().sum())
print(df['Sumber Tanggal'].value_counts(dropna=False))

In [ ]:
df['Tanggal Analisis'] = pd.to_datetime(
    df['Tanggal Analisis'],
    errors='coerce'
)

In [ ]:
print('Sesudah Preprocessing:')
print('Tipe Data:', df['Tanggal Analisis'].dtype)
print('Jumlah Missing Value:', df['Tanggal Analisis'].isnull().sum())
print(df['Tanggal Analisis'].head())

Catatan :
- Kolom `Tanggal Analisis` dikonversi menjadi tipe data datetime agar dapat digunakan untuk pengelompokan berdasarkan bulan.
- Untuk sebagian besar dataset, tanggal analisis berasal dari `Waktu Pesanan Dibuat`.
- Pada file Desember 2024 dan Juli 2025, tanggal analisis menggunakan `Waktu Pengiriman Diatur` karena kolom waktu pesanan tidak tersedia.
- Baris yang tidak memiliki kedua informasi tanggal tetap menjadi missing value dan tidak diisi dengan tanggal perkiraan.

## (5.5) Preprocessing pada kolom Total Pembayaran

In [ ]:
print(' Sebelum Preprocessing : \n')
print('Tipe Data :', df['Total Pembayaran'].dtype)
pd.Series(df['Total Pembayaran'].unique())

In [ ]:
df['Total Pembayaran'] = (
    df['Total Pembayaran']
    .astype(str)
    .str.replace('.', '', regex=False)
    .astype(float)
)

In [ ]:
print(' Sesudah Preprocessing : \n')
print('Tipe Data :', df['Total Pembayaran'].dtype)
print('Missing Value :', df['Total Pembayaran'].isnull().sum())
df['Total Pembayaran'].head()

## (5.6) Pemeriksaan pada kolom Status Pesanan

In [ ]:
print('Jumlah Status Pesanan :')
print(df['Status Pesanan'].value_counts(dropna=False))

In [ ]:
df['Status Pesanan Ringkas'] = df['Status Pesanan'].apply(
    lambda x: 'Pesanan Diterima'
    if isinstance(x, str) and x.startswith('Pesanan diterima')
    else x
)

print(df['Status Pesanan Ringkas'].value_counts(dropna=False))

Catatan :
- Ditemukan 11 variasi status pesanan pada dataset.
- Beberapa status memiliki keterangan tanggal batas pengajuan pengembalian.
- Dilakukan penyederhanaan status dengan membuat kolom baru `Status Pesanan Ringkas`.
- Status dengan awalan "Pesanan diterima" dikelompokkan menjadi "Pesanan Diterima".
- Kolom status asli tetap dipertahankan agar informasi awal tidak hilang.

## (5.7) Pemeriksaan pada kolom Metode Pembayaran

In [ ]:
print(df['Metode Pembayaran'].value_counts(dropna=False))

Catatan :
- Terdapat 12 kategori metode pembayaran pada dataset.
- Metode pembayaran yang paling banyak digunakan adalah COD (Bayar di Tempat), yaitu 13.153 transaksi.
- Tidak ditemukan nilai kosong pada kolom Metode Pembayaran.
- Data dipertahankan tanpa perubahan karena kategorinya sudah dapat digunakan untuk analisis.

## (5.8) Pemeriksaan pada kolom Alasan Pembatalan

In [ ]:
print(df['Alasan Pembatalan'].value_counts(dropna=False))

Catatan :
- Ditemukan 18 variasi alasan pembatalan dengan total 3.525 transaksi.
- Nilai kosong pada kolom Alasan Pembatalan berjumlah 22.733.
- Nilai kosong tersebut berasal dari pesanan yang tidak dibatalkan sehingga tidak memiliki alasan pembatalan.
- Data dipertahankan tanpa mengisi nilai kosong agar informasi tetap sesuai kondisi transaksi sebenarnya.

## (5.9) Pengelompokan Pihak Pembatalan

In [ ]:
df['Pihak Pembatalan'] = df['Alasan Pembatalan'].str.extract(
    r'Dibatalkan (?:secara otomatis oleh |oleh )(Pembeli|Penjual|sistem)',
    expand=False
)

print(df['Pihak Pembatalan'].value_counts(dropna=False))

Catatan :
- Informasi pihak pembatalan diekstrak dari kolom `Alasan Pembatalan`.
- Pihak pembatalan dikelompokkan menjadi Pembeli, Penjual, dan sistem.
- Nilai kosong pada kolom ini menunjukkan pesanan yang tidak memiliki informasi pihak pembatalan.
- Kolom `Alasan Pembatalan` asli tetap dipertahankan.

## (5.10) Pemeriksaan pada kolom product_category

In [ ]:
print(df['product_category'].value_counts(dropna=False))

In [ ]:
print('Missing Value :', df['product_category'].isnull().sum())
print('Jumlah Kategori :', df['product_category'].nunique())

In [ ]:
print(df['product_category'].value_counts())

Catatan :
- Kolom product_category memiliki 38 kategori produk unik.
- Tidak ditemukan missing value pada kolom ini.
- Kategori dengan jumlah transaksi terbanyak adalah Celengan sebanyak 6.999 transaksi.
- Data dipertahankan tanpa perubahan karena sudah dapat digunakan untuk analisis kategori produk.

## (5.11) Pemeriksaan dan Penghapusan Data Duplikat

In [ ]:
# Memeriksa dan Menghapus Data Duplikat

print('Jumlah data duplikat:', df.duplicated().sum())

df.drop_duplicates(inplace=True)

print('Jumlah data setelah menghapus duplikasi:', df.shape)
print('Jumlah data duplikat setelah dibersihkan:', df.duplicated().sum())

Catatan :
- Pemeriksaan dilakukan untuk mengidentifikasi baris yang memiliki nilai sama pada seluruh kolom.
- Baris duplikat penuh dihapus agar tidak terjadi penghitungan berulang.
- Baris dengan `order_id` sama tetapi memiliki kategori produk berbeda tetap dipertahankan.
- Setelah proses ini, dataset digunakan untuk tahap Exploratory Data Analysis dan Analysis Process.

#  (6) Exploratory Data Analysis or EDA

## (6.1) Exploratory Analysis using Basic Statistical Approach

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
# Mendeteksi nilai anomali pada data secara manual

col = df.columns

for i in col:
    if df[i].dtype == 'object':
        print('Kolom {} berisikan nilai :'.format(str(i)))
        print(df[i].unique())
        print(' ')

Catatan :
- Statistik deskriptif digunakan untuk memahami sebaran, nilai rata-rata, median, minimum, maksimum, dan karakteristik variabel.
- Pemeriksaan nilai unik pada kolom kategorikal membantu mengetahui variasi kategori dan mendeteksi perbedaan penulisan.
- Jumlah baris pada dataset merepresentasikan baris data produk/transaksi, sehingga tidak selalu sama dengan jumlah `order_id` unik.

## (6.2) Univariate Analysis

In [ ]:
# Distribusi Data Total Pembayaran

plt.figure(figsize=(15, 8))
plt.title('Distribusi Data Total Pembayaran')
sns.histplot(df['Total Pembayaran'], kde=True)
plt.xlabel('Total Pembayaran')
plt.ylabel('Frekuensi')
plt.show()

In [ ]:
# Distribusi Data Total Diskon

plt.figure(figsize=(15, 8))
plt.title('Distribusi Data Total Diskon')
sns.histplot(df['Total Diskon'], kde=True)
plt.xlabel('Total Diskon')
plt.ylabel('Frekuensi')
plt.show()

In [ ]:
# Distribusi Data Jumlah Produk

plt.figure(figsize=(15, 8))
plt.title('Distribusi Data Jumlah Produk')
sns.histplot(df['Jumlah'], kde=True)
plt.xlabel('Jumlah Produk')
plt.ylabel('Frekuensi')
plt.show()

In [ ]:
# Distribusi Status Pesanan

plt.figure(figsize=(12, 6))
plt.title('Distribusi Status Pesanan')
sns.countplot(data=df, x='Status Pesanan Ringkas')
plt.xlabel('Status Pesanan')
plt.ylabel('Frekuensi')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Distribusi Metode Pembayaran

plt.figure(figsize=(14, 7))
plt.title('Distribusi Metode Pembayaran')
sns.countplot(
    data=df,
    y='Metode Pembayaran',
    order=df['Metode Pembayaran'].value_counts().index
)
plt.xlabel('Frekuensi')
plt.ylabel('Metode Pembayaran')
plt.show()

In [ ]:
# Distribusi Kategori Produk

plt.figure(figsize=(14, 10))
plt.title('Distribusi Kategori Produk')
sns.countplot(
    data=df,
    y='product_category',
    order=df['product_category'].value_counts().index
)
plt.xlabel('Frekuensi')
plt.ylabel('Kategori Produk')
plt.show()

In [ ]:
# Distribusi Transaksi Berdasarkan Provinsi

plt.figure(figsize=(14, 10))
plt.title('Distribusi Transaksi Berdasarkan Provinsi')
sns.countplot(
    data=df,
    y='Provinsi',
    order=df['Provinsi'].value_counts().index
)
plt.xlabel('Frekuensi')
plt.ylabel('Provinsi')
plt.show()

In [ ]:
# Distribusi Alasan Pembatalan

plt.figure(figsize=(14, 9))
plt.title('Distribusi Alasan Pembatalan')

sns.countplot(
    data=df.dropna(subset=['Alasan Pembatalan']),
    y='Alasan Pembatalan',
    order=df['Alasan Pembatalan'].value_counts().index
)

plt.xlabel('Frekuensi')
plt.ylabel('Alasan Pembatalan')
plt.show()

In [ ]:
# Distribusi Data Jumlah Produk

plt.figure(figsize=(15, 8))
plt.title('Distribusi Data Pada Jumlah Produk')
sns.kdeplot(df['Jumlah'])
plt.xlabel('Jumlah Produk')
plt.show()

In [ ]:
# Distribusi Data Total Pembayaran

plt.figure(figsize=(15, 8))
plt.title('Distribusi Data Pada Total Pembayaran')
sns.kdeplot(df['Total Pembayaran'])
plt.xlabel('Total Pembayaran')
plt.show()

Catatan :
- Analisis univariat digunakan untuk melihat distribusi masing-masing variabel.
- Distribusi Total Pembayaran, Total Diskon, dan Jumlah Produk menunjukkan konsentrasi data pada nilai yang relatif rendah serta beberapa nilai yang jauh lebih tinggi.
- Kategori produk, provinsi, metode pembayaran, status pesanan, dan alasan pembatalan memiliki frekuensi yang bervariasi.
- Frekuensi pada grafik univariat dihitung berdasarkan baris data setelah penghapusan duplikasi penuh, bukan selalu berdasarkan jumlah pesanan unik.

## (6.3) Bivariate Analysis

In [ ]:
# Melakukan Bivariate Analysis pada data bertipe numerik

numeric_cols = [
    'Jumlah',
    'Total Diskon',
    'Total Pembayaran',
    'Total Berat'
]

g = sns.pairplot(df[numeric_cols], kind='scatter', height=3)
g.fig.suptitle('Bivariate Analysis - Numerical Variables', y=1.02)
plt.show()

In [ ]:
# Melihat Korelasi Antarvariabel Numerik

plt.figure(figsize=(10, 8))

sns.heatmap(
    df[['Jumlah', 'Total Diskon', 'Total Pembayaran', 'Total Berat']].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.show()

Catatan :
- Pairplot dan heatmap digunakan untuk mengamati pola hubungan antarvariabel numerik.
- Hasil korelasi menunjukkan hubungan antarvariabel memiliki kekuatan yang berbeda.
- Korelasi hanya menunjukkan hubungan statistik dan tidak membuktikan hubungan sebab-akibat.
- Nilai pembayaran merupakan nilai pada tingkat pesanan yang dapat muncul pada beberapa baris kategori dalam satu pesanan. Oleh karena itu, analisis yang menggunakan total pembayaran dilakukan dengan satu baris per `order_id`.

# (7) Analysis Process

## (7.1) Kategori Produk dengan Jumlah Pesanan Terbanyak

In [ ]:
# Kategori Produk dengan Jumlah Pesanan Terbanyak

df1 = df.groupby(['product_category'])['order_id'].nunique().to_frame()
df1 = df1.sort_values('order_id', ascending=False)
df1.rename(columns={'order_id': 'Jumlah Pesanan'}, inplace=True)
df1.reset_index(inplace=True)
df1.index += 1

topCategory = df1.head(10)
topCategory

## (7.2) Tren Total Pembayaran Berdasarkan Bulan

In [ ]:
# Menggunakan satu baris untuk setiap order_id agar pembayaran tidak dihitung berulang

df2 = df.drop_duplicates(subset='order_id').copy()

df2['Bulan'] = df2['Tanggal Analisis'].dt.to_period('M')

df2 = df2.groupby('Bulan')['Total Pembayaran'].sum().to_frame()
df2.reset_index(inplace=True)

# Mengubah period menjadi datetime agar dapat divisualisasikan
df2['Bulan'] = df2['Bulan'].dt.to_timestamp()

df2

In [ ]:
plt.figure(figsize=(15, 6))

sns.lineplot(
    data=df2,
    x='Bulan',
    y='Total Pembayaran',
    marker='o'
)

plt.title('Tren Total Pembayaran per Bulan')
plt.xlabel('Bulan')
plt.ylabel('Total Pembayaran (Rp)')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## (7.3) Provinsi dengan Total Pembayaran Tertinggi

In [ ]:
# Provinsi dengan Total Pembayaran Tertinggi

df3 = df.drop_duplicates(subset='order_id').copy()

df3 = df3.groupby(['Provinsi'])['Total Pembayaran'].sum().to_frame()
df3 = df3.sort_values('Total Pembayaran', ascending=False)
df3.rename(columns={'Total Pembayaran': 'Total Penjualan'}, inplace=True)
df3.reset_index(inplace=True)
df3.index += 1

topProvince = df3.head(10)
topProvince

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=topProvince,
    x='Total Penjualan',
    y='Provinsi'
)

plt.title('Top 10 Provinsi berdasarkan Total Pembayaran')
plt.xlabel('Total Pembayaran (Rp)')
plt.ylabel('Provinsi')
plt.tight_layout()
plt.show()

## (7.4) Hubungan Total Diskon dan Total Pembayaran

In [ ]:
# Analisis Hubungan Total Diskon dan Total Pembayaran

df4 = df.drop_duplicates(subset='order_id').copy()

df4[['Total Diskon', 'Total Pembayaran']].corr()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df4,
    x='Total Diskon',
    y='Total Pembayaran'
)

plt.title('Hubungan Total Diskon dan Total Pembayaran')
plt.xlabel('Total Diskon')
plt.ylabel('Total Pembayaran (Rp)')
plt.tight_layout()
plt.show()

## (7.5) Metode Pembayaran yang Paling Banyak Digunakan

In [ ]:
# Metode Pembayaran yang Paling Banyak Digunakan

df5 = df.drop_duplicates(subset='order_id').copy()

df5 = df5.groupby(['Metode Pembayaran'])['order_id'].nunique().to_frame()
df5 = df5.sort_values('order_id', ascending=False)
df5.rename(columns={'order_id': 'Jumlah Pesanan'}, inplace=True)
df5.reset_index(inplace=True)
df5.index += 1

df5

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=df5,
    x='Jumlah Pesanan',
    y='Metode Pembayaran'
)

plt.title('Jumlah Pesanan Berdasarkan Metode Pembayaran')
plt.xlabel('Jumlah Pesanan')
plt.ylabel('Metode Pembayaran')
plt.tight_layout()
plt.show()

## (7.6) Jumlah Pesanan Berdasarkan Pihak Pembatalan

In [ ]:
# Jumlah Pesanan Berdasarkan Pihak Pembatalan

df6 = df.drop_duplicates(subset='order_id').copy()

df6 = df6.groupby(['Pihak Pembatalan'])['order_id'].nunique().to_frame()
df6 = df6.sort_values('order_id', ascending=False)
df6.rename(columns={'order_id': 'Jumlah Pesanan'}, inplace=True)
df6.reset_index(inplace=True)
df6.index += 1

df6

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=df6,
    x='Jumlah Pesanan',
    y='Pihak Pembatalan'
)

plt.title('Jumlah Pesanan Berdasarkan Pihak Pembatalan')
plt.xlabel('Jumlah Pesanan')
plt.ylabel('Pihak Pembatalan')
plt.tight_layout()
plt.show()

# (8) Insight and Information based on Analysis Result

**Kesimpulan :**

- Berdasarkan hasil analisis tren total pembayaran E-Commerce selama periode Desember 2023 hingga November 2025, diperoleh beberapa temuan:
  - **September 2024** merupakan bulan dengan total pembayaran tertinggi sebesar **Rp71.193.097**.
  - **Mei 2025** merupakan bulan dengan total pembayaran terendah sebesar **Rp29.139.406**.
  - Setelah mengalami penurunan hingga Mei 2025, total pembayaran kembali meningkat dan mencapai **Rp61.464.654** pada Oktober 2025.

- Berdasarkan kategori produk dengan jumlah pesanan terbanyak, terdapat 10 kategori produk dengan jumlah pesanan tertinggi, yaitu:
  1. **Celengan** dengan 5.869 pesanan (Top 1).
  2. **Mangkok Sambal / Saus** dengan 3.952 pesanan (Top 2).
  3. **Aksesoris Pintu** dengan 2.962 pesanan (Top 3).
  4. **Nampan / Tray** dengan 1.514 pesanan.
  5. **Other** dengan 898 pesanan.
  6. **Baskom / Mangkok Besar** dengan 760 pesanan.
  7. **Keranjang** dengan 718 pesanan.
  8. **Seal / Baut / Roof** dengan 690 pesanan.
  9. **Rak / Rak Serbaguna** dengan 663 pesanan.
  10. **Lunch Box / Rantang** dengan 569 pesanan.

- Berdasarkan wilayah atau provinsi dengan total pembayaran tertinggi, diperoleh 10 provinsi dengan nilai transaksi terbesar, yaitu:
  1. **Jawa Barat** sebesar Rp276.992.073 (Top 1).
  2. **Banten** sebesar Rp177.861.752 (Top 2).
  3. **DKI Jakarta** sebesar Rp135.164.841 (Top 3).
  4. **Jawa Tengah** sebesar Rp100.114.044.
  5. **Jawa Timur** sebesar Rp95.478.924.
  6. **Sumatera Selatan** sebesar Rp40.909.428.
  7. **Lampung** sebesar Rp22.028.652.
  8. **Bali** sebesar Rp21.951.090.
  9. **Sumatera Utara** sebesar Rp19.783.218.
  10. **Sulawesi Selatan** sebesar Rp19.680.129.

- Berdasarkan analisis diskon, metode pembayaran, dan pembatalan pesanan, terdapat beberapa temuan:
  - Hubungan antara **Total Diskon dan Total Pembayaran** memiliki korelasi positif lemah sebesar **0,210377**.
  - Metode pembayaran yang paling banyak digunakan adalah **COD (Bayar di Tempat)** dengan 11.538 pesanan, diikuti Saldo ShopeePay sebanyak 3.692 pesanan dan Online Payment sebanyak 3.170 pesanan.
  - Berdasarkan pihak pembatalan, **Pembeli** menjadi pihak dengan jumlah pembatalan tertinggi sebanyak 1.912 pesanan, diikuti sistem sebanyak 904 pesanan dan Penjual sebanyak 14 pesanan.

- Berdasarkan hasil analisis data E-Commerce tersebut, terdapat beberapa insight dan rekomendasi yang dapat dijadikan sebagai

## (9) Exporting Data

In [ ]:
with pd.ExcelWriter('Ecommerce_Sales_Analysis.xlsx') as writer:
    df.to_excel(writer, sheet_name='Original', index=False)
    topCategory.to_excel(writer, sheet_name='Top Category', index=False)
    df2.to_excel(writer, sheet_name='Monthly Payment', index=False)
    topProvince.to_excel(writer, sheet_name='Top Province', index=False)
    df5.to_excel(writer, sheet_name='Payment Method', index=False)
    df6.to_excel(writer, sheet_name='Cancellation', index=False)